# 06 · H2: the English blind spot

Narrative only: the logic lives in `src/analysis/h2_blindspot.py`. Run `python run.py analyse` first.

**Pre-specified design (D-24, D-53, D-54).** Population: colexification edges (≥ 2 families) in the analysis set, minus same-collision-group pairs. Groups from the audited English gold standard. Outcome: mutual nearest-neighbour rank. Confirmatory test: one-sided Mann–Whitney U on the 1:3 propensity-matched sample (exact on semantic field, caliper 0.2 SD), Cliff's δ with BCa CI, Benjamini–Hochberg across conditions. The group × model interaction needs ≥ 2 conditions.

In [ ]:
import json, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
import config
from IPython.display import Image
load = lambda name: json.loads((config.RESULTS_DIR / name).read_text())

## 1. The English gold standard (audit)
`decision` (human) overrides Claude's `suggested_decision`. Until every row has a human decision, results are labelled PROVISIONAL.

In [ ]:
a = pd.read_csv(config.ENGLISH_AUDIT_PATH, keep_default_na=False)
print('human decisions:', (a['decision'] != '').sum(), 'of', len(a))
a['suggested_decision'].value_counts()

In [ ]:
a[a['suggested_decision'] == 'present'][['gloss_a','gloss_b','n_families','notes']]

## 2. Why matching is non-negotiable
English-present colexifications are far more widespread than English-absent ones (the opposite of what D-24 anticipated), so an unmatched comparison is confounded.

In [ ]:
d = load('h2_design.json'); d['populations'], d['balanced_after_matching']

In [ ]:
Image(filename=str(config.FIGURES_DIR / 'fig6_h2_balance.png'), width=900)

## 3. Confirmatory result

In [ ]:
r = pd.read_csv(config.RESULTS_DIR / 'h2_blindspot.csv')
r[r['status'] == 'done'][['label','design','population','primary','n_present','n_absent','median_rank_present','median_rank_absent','cliffs_delta','delta_ci_low','delta_ci_high','p_one_sided','audit_status']]

In [ ]:
Image(filename=str(config.FIGURES_DIR / 'fig7_h2_ranks.png'), width=560)

## 4. How much does the result depend on the matching? (D-54)

In [ ]:
pd.read_csv(config.RESULTS_DIR / 'h2_matching_sensitivity.csv')

## 5. Model by model (the interaction)

In [ ]:
Image(filename=str(config.FIGURES_DIR / 'fig8_h2_gap_by_model.png'), width=820)

In [ ]:
p = config.RESULTS_DIR / 'h2_interaction.csv'
pd.read_csv(p) if p.exists() else d.get('interaction')

## 6. Case study: *kal* and the named pairs
Neighbour rank 0 = each concept is the other's nearest neighbour. `input_colexified` marks pairs that one of the five input languages itself colexifies (D-49).

In [ ]:
cs = pd.read_csv(config.RESULTS_DIR / 'h2_case_study.csv')
cs[['condition','pair','english','families','cosine','mutual_rank','input_colexified']]

In [ ]:
cs.filter(like='neighbours_').assign(pair=cs['pair'], condition=cs['condition']).set_index(['condition','pair']).dropna(how='all', axis=1)